In [1]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.3 MB/s eta 0:00:00


In [2]:
pip install ensemble-boxes

In [3]:
from google.colab import drive
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
import os
import numpy as np
from torchvision import transforms
from ultralytics import YOLO
from IPython.display import Image, display
from ultralytics import RTDETR
import gc; import torch
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import urllib.request
from ensemble_boxes import weighted_boxes_fusion
import json

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
zip_path = '/content/drive/MyDrive/new_dataset.zip'
extract_path = '/content/yolo_dataset'

!unzip -q -o "{zip_path}" -d "{extract_path}"

In [6]:
#경로설정
train_img_path = os.path.join(extract_path, 'train', 'images')
train_label_path = os.path.join(extract_path, 'train', 'labels')
val_img_path = os.path.join(extract_path, 'val', 'images')
val_label_path = os.path.join(extract_path, 'val', 'labels')
test_img_path = os.path.join(extract_path, 'test', 'images')

paths = {
    "Train 이미지": train_img_path,
    "Train 라벨": train_label_path,
    "Val 이미지": val_img_path,
    "Val 라벨": val_label_path,
    "Test 이미지": test_img_path
}

for name, path in paths.items():
    if os.path.exists(path):
        count = len(os.listdir(path))
        print(f"{name}: {count}개")
    else:
        print(f"{name}: 경로 없음")

# 최종 합계 (이미지 기준)
total_images = len(os.listdir(train_img_path)) + len(os.listdir(val_img_path))
print(f"총 이미지 합계: {total_images}개")

Train 이미지: 295개
Train 라벨: 295개
Val 이미지: 37개
Val 라벨: 37개
Test 이미지: 842개
총 이미지 합계: 332개


In [ ]:
#디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
model = YOLO('yolo11m.pt')


results = model.train(
    data=r"/content/yolo_dataset/data.yaml",
    epochs=30,
    imgsz=1024,
    batch=8,
    device=device,
    project='pill_project',
    name='yolov11m',

    patience=15,
    cos_lr=True,
    degrees=0,
    mosaic=1.0,
)

engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov11m, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=15, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, project=pill_project,

In [ ]:
# 1. 모델 로드 및 검증 실행
best_model_yolo = YOLO('/content/runs/detect/pill_project/yolov11m/weights/best.pt')
metrics_yolo = best_model_yolo.val(split='val')

# 2. 전체 평균 지표 (Summary)
map50 = metrics_yolo.box.map50
map50_95 = metrics_yolo.box.map
precision = metrics_yolo.box.mp
recall = metrics_yolo.box.mr

print(f"🚀 YOLO11m 검증 결과 요약")
print(f"✅ 정밀도 (Precision)   : {precision:.4f}")
print(f"✅ 재현율 (Recall)      : {recall:.4f}")
print(f"✅ mAP @ 0.5           : {map50:.4f}")
print(f"✅ mAP @ 0.5:0.95      : {map50_95:.4f}")

print("\n📋 [클래스별 상세 지표]")
for i, class_idx in enumerate(metrics_yolo.box.ap_class_index):
    class_name = best_model_yolo.names[class_idx]
    p = metrics_yolo.box.p[i]
    r = metrics_yolo.box.r[i]
    m50 = metrics_yolo.box.ap50[i]
    m95 = metrics_yolo.box.ap[i]

    print(f"[{class_name}] P:{p:.3f} | R:{r:.3f} | mAP50:{m50:.3f} | mAP95:{m95:.3f}")

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 126 layers, 20,073,208 parameters, 0 gradients, 67.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4167.0±1594.1 MB/s, size: 1724.0 KB)
val: Scanning /content/yolo_dataset/val/labels.cache... 37 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 37/37 11.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.5s/it 4.6s
                   all         37        129       0.95      0.994      0.991      0.982
            보령부스파정 5mg          5          5      0.984          1      0.995      0.995
           가바토파정 100mg          6          6      0.817      0.833      0.772      0.772
             스토가정 10mg          2          2      0.955          1      0.995      0.995
                  레일라정          1          1      0.934          1      0.995      0.995
                  신바로정          1          

In [9]:
MODEL_PATH = '/content/drive/MyDrive/best_YOLO11m.pt'
TEST_DIR = '/content/yolo_dataset/test/images'
SAVE_NAME = 'yolov11m_submission.csv'

true_class_ids = [1900, 16548, 19607, 29451, 33009, 21771, 27926, 24850, 29345, 16551, 33208, 2483, 3743, 12778, 13395, 12081, 25438, 19552, 22362, 3351, 3832, 16232, 16262, 16688, 20238, 21325, 22074, 29667, 35206, 36637, 38162, 13900, 18357, 19232, 20014, 31863, 32310, 33880, 41768, 18147, 3483, 20877, 30308, 34597, 22347, 25469, 19861, 28763, 27733, 25367, 31885, 27777, 3544, 4543, 12247, 27993]
model = YOLO(MODEL_PATH)
results = model.predict(source=TEST_DIR, conf=0.01, augment=True ,save=False) # 추론 시작

final_rows = []
annotation_id = 1

for res in results:
    file_name = os.path.basename(res.path)
    image_id = int(''.join(filter(str.isdigit, file_name)))

    boxes = res.boxes
    for box in boxes:

        cls_idx = int(box.cls[0])


        category_id = true_class_ids[cls_idx]

        xyxy = box.xyxy[0].cpu().numpy()
        bbox_x = int(xyxy[0])
        bbox_y = int(xyxy[1])
        bbox_w = int(xyxy[2] - xyxy[0])
        bbox_h = int(xyxy[3] - xyxy[1])

        score = round(float(box.conf[0]), 2)

        final_rows.append([
            annotation_id, image_id, category_id,
            bbox_x, bbox_y, bbox_w, bbox_h, score
        ])
        annotation_id += 1

df = pd.DataFrame(final_rows, columns=[
    'annotation_id', 'image_id', 'category_id',
    'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'score'
])

df.to_csv(SAVE_NAME, index=False)
print(f"작업 완료")


image 1/842 /content/yolo_dataset/test/images/test_images_1.png: 1024x800 1 보령부스파정 5mg, 2 울트라셋이알서방정s, 1 비모보정 500/20mg, 1 동아가바펜틴정 800mg, 1 카발린캡슐 25mg, 1 에어탈정(아세클로페낙), 223.5ms
image 2/842 /content/yolo_dataset/test/images/test_images_10.png: 1024x800 1 보령부스파정 5mg, 1 가바토파정 100mg, 1 레일라정, 1 라비에트정 20mg, 1 엑스포지정 5/160mg, 1 카나브정 60mg, 102.7ms
image 3/842 /content/yolo_dataset/test/images/test_images_100.png: 1024x800 1 보령부스파정 5mg, 1 가바토파정 100mg, 1 신바로정, 1 마도파정, 1 아빌리파이정 10mg, 92.3ms
image 4/842 /content/yolo_dataset/test/images/test_images_1003.png: 1024x800 1 리피토정 20mg, 1 기넥신에프정(은행엽엑스)(수출용), 1 제미메트서방정 50/1000mg, 1 트윈스타정 40/5mg, 92.1ms
image 5/842 /content/yolo_dataset/test/images/test_images_1004.png: 1024x800 1 리피토정 20mg, 1 기넥신에프정(은행엽엑스)(수출용), 1 제미메트서방정 50/1000mg, 1 트윈스타정 40/5mg, 92.2ms
image 6/842 /content/yolo_dataset/test/images/test_images_1005.png: 1024x800 1 리피토정 20mg, 1 기넥신에프정(은행엽엑스)(수출용), 1 제미메트서방정 50/1000mg, 1 트윈스타정 40/5mg, 91.8ms
image 7/842 /content/yolo_dataset/test/images/test